# 02 — National-Level EDA

Comprehensive exploratory data analysis of national food surplus/waste data.

**Outputs:**
- `outputs/analysis/eda_report.json` — comprehensive statistics
- `outputs/analysis/charts/` — 9 publication-quality PNGs

**Sections:**
1. Load Data
2. EDA Statistics → eda_report.json
3. Charts (9 charts)

In [1]:
import os, sys
# Ensure we're at project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import warnings
warnings.filterwarnings('ignore')
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

CHARTS = 'outputs/analysis/charts'
os.makedirs(CHARTS, exist_ok=True)
os.makedirs('outputs/analysis', exist_ok=True)

COLORS = {'Farm': '#2C5F2D', 'Foodservice': '#F96167', 'Manufacturing': '#065A82',
          'Residential': '#F9E795', 'Retail': '#028090'}
SECTOR_ORDER = ['Farm', 'Foodservice', 'Manufacturing', 'Residential', 'Retail']

def savefig(name):
    plt.savefig(f'{CHARTS}/{name}', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f'Saved {name}')

---
## Part 1: Load Data

In [2]:
from utils.data_loader import load_all

data = load_all()
ns = data['national_summary']
nc = data['national_cause']
ns24 = ns[ns['year'] == 2024]
nc24 = nc[nc['year'] == 2024]
print(f"National summary: {ns.shape}, National cause: {nc.shape}")
print(f"2024 slices: summary={ns24.shape}, cause={nc24.shape}")

National summary: (4350, 31), National cause: (16455, 10)
2024 slices: summary=(290, 31), cause=(1097, 10)


---
## Part 2: EDA Statistics → eda_report.json

In [3]:
report = {}

# Overview 2024
o = ns24.groupby('year').agg(
    total_surplus_tons=('tons_surplus', 'sum'),
    total_waste_tons=('tons_waste', 'sum'),
    total_dollars=('us_dollars_surplus', 'sum'),
    total_landfill_tons=('tons_landfill', 'sum'),
    total_donations_tons=('tons_donations', 'sum'),
    total_composting_tons=('tons_composting', 'sum'),
    total_co2e_mt=('surplus_total_100_year_mtco2e_footprint', 'sum'),
    total_water_gallons=('gallons_water_footprint', 'sum'),
    total_meals_wasted=('meals_wasted', 'sum'),
).iloc[0].to_dict()
report['overview_2024'] = {k: round(v, 2) for k, v in o.items()}

# Sector breakdown 2024
sec = ns24.groupby('sector').agg(
    tons_surplus=('tons_surplus', 'sum'), tons_waste=('tons_waste', 'sum'),
    tons_landfill=('tons_landfill', 'sum'), tons_donations=('tons_donations', 'sum'),
    us_dollars=('us_dollars_surplus', 'sum'),
    co2e_mt=('surplus_total_100_year_mtco2e_footprint', 'sum'),
).reset_index()
total_surplus = sec['tons_surplus'].sum()
total_landfill = sec['tons_landfill'].sum()
sec['pct_of_surplus'] = round(sec['tons_surplus'] / total_surplus * 100, 2)
sec['pct_of_landfill'] = round(sec['tons_landfill'] / total_landfill * 100, 2)
report['sector_breakdown_2024'] = sec.set_index('sector').to_dict('index')

# Food type breakdown 2024
ft = ns24.groupby('food_type').agg(
    tons_surplus=('tons_surplus', 'sum'), tons_waste=('tons_waste', 'sum'),
    tons_landfill=('tons_landfill', 'sum')).reset_index()
ft['pct'] = round(ft['tons_surplus'] / ft['tons_surplus'].sum() * 100, 2)
report['food_type_breakdown_2024'] = ft.set_index('food_type').to_dict('index')

print(f"Overview 2024: {len(report['overview_2024'])} metrics")
print(f"Sectors: {list(report['sector_breakdown_2024'].keys())}")
print(f"Food types: {len(report['food_type_breakdown_2024'])} categories")

Overview 2024: 9 metrics
Sectors: ['Farm', 'Foodservice', 'Manufacturing', 'Residential', 'Retail']
Food types: 8 categories


In [4]:
# Destination breakdown 2024
dest_cols = ['tons_landfill', 'tons_composting', 'tons_donations', 'tons_animal_feed',
             'tons_anaerobic_digestion', 'tons_sewer', 'tons_dumping', 'tons_incineration',
             'tons_land_application', 'tons_not_harvested', 'tons_industrial_uses']
total_waste = ns24['tons_waste'].sum()
report['destination_breakdown_2024'] = {
    c.replace('tons_', ''): {'tons': round(ns24[c].sum(), 2),
                             'pct_of_waste': round(ns24[c].sum() / total_waste * 100, 2)}
    for c in dest_cols}

# Cause breakdown 2024
cg = nc24.groupby('cause_group')['tons_surplus_due_to_cause'].sum().sort_values(ascending=False)
cn = nc24.groupby('cause_name')['tons_surplus_due_to_cause'].sum().sort_values(ascending=False).head(15)
report['cause_breakdown_2024'] = {
    'by_group': {k: round(v, 2) for k, v in cg.items()},
    'by_name_top15': {k: round(v, 2) for k, v in cn.items()},
}

# Sector x Cause 2024
sxc = nc24.groupby(['sector', 'cause_name'])['tons_surplus_due_to_cause'].sum().reset_index()
report['sector_x_cause_2024'] = {}
for sector in sxc['sector'].unique():
    sub = sxc[sxc['sector'] == sector].nlargest(5, 'tons_surplus_due_to_cause')
    tot = sub['tons_surplus_due_to_cause'].sum()
    report['sector_x_cause_2024'][sector] = [
        {'cause': r['cause_name'], 'tons': round(r['tons_surplus_due_to_cause'], 2),
         'pct': round(r['tons_surplus_due_to_cause'] / tot * 100, 2)}
        for _, r in sub.iterrows()]

print(f"Destinations: {len(report['destination_breakdown_2024'])} types")
print(f"Cause groups: {len(report['cause_breakdown_2024']['by_group'])}")

Destinations: 11 types
Cause groups: 9


In [5]:
# Trend 2010-2024
yearly = ns.groupby('year').agg(total_surplus=('tons_surplus', 'sum'),
    total_waste=('tons_waste', 'sum'), total_landfill=('tons_landfill', 'sum')).reset_index()
by_sec_yr = ns.groupby(['year', 'sector']).agg(surplus=('tons_surplus', 'sum'),
    waste=('tons_waste', 'sum'), landfill=('tons_landfill', 'sum')).reset_index()
report['trend_2010_2024'] = {
    'yearly_totals': yearly.to_dict('records'),
    'by_sector': {s: g[['year', 'surplus', 'waste', 'landfill']].to_dict('records')
                  for s, g in by_sec_yr.groupby('sector')},
}

# Waste intensity 2024
wi = ns24.groupby(['sector', 'food_type']).agg(w=('tons_waste', 'sum'), s=('tons_supply', 'sum')).reset_index()
wi['intensity'] = wi['w'] / wi['s']
report['waste_intensity_2024'] = wi.pivot(index='food_type', columns='sector',
    values='intensity').fillna(0).round(4).to_dict()

# Environmental impact 2024
env_sec = ns24.groupby('sector').agg(
    co2e=('surplus_total_100_year_mtco2e_footprint', 'sum'),
    water=('gallons_water_footprint', 'sum')).to_dict('index')
total_co2e = ns24['surplus_total_100_year_mtco2e_footprint'].sum()
report['environmental_impact_2024'] = {
    'total_co2e_mt': round(total_co2e, 2),
    'total_water_gallons': round(ns24['gallons_water_footprint'].sum(), 2),
    'by_sector': {k: {kk: round(vv, 2) for kk, vv in v.items()} for k, v in env_sec.items()},
    'co2e_per_ton': round(total_co2e / total_surplus, 4),
    'car_equivalents': round(total_co2e / 4.6, 0),
}

print(f"Trend years: {len(report['trend_2010_2024']['yearly_totals'])}")
print(f"Environmental: {report['environmental_impact_2024']['total_co2e_mt']:.0f} MT CO2e total")

Trend years: 15
Environmental: 221012600 MT CO2e total


In [6]:
# Donation gap
dg = ns24.groupby('sector').agg(total_uneaten=('tons_uneaten', 'sum'),
    total_donated=('tons_donations', 'sum')).reset_index()
dg['donation_rate_pct'] = round(dg['total_donated'] / dg['total_uneaten'] * 100, 4)
report['donation_gap_analysis'] = dg.set_index('sector').to_dict('index')

# COVID impact
for y in [2019, 2020, 2021]:
    ydf = ns[ns['year'] == y]
    report.setdefault('covid_impact', {})[str(y)] = {
        'total_surplus': round(ydf['tons_surplus'].sum(), 2),
        'total_waste': round(ydf['tons_waste'].sum(), 2)}
s19, s20, s21 = [report['covid_impact'][str(y)]['total_surplus'] for y in [2019, 2020, 2021]]
report['covid_impact']['drop_pct_2019_2020'] = round((s20 - s19) / s19 * 100, 2)
report['covid_impact']['recovery_pct_2020_2021'] = round((s21 - s20) / s20 * 100, 2)

# Save JSON
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)): return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, (np.ndarray,)): return obj.tolist()
        return super().default(obj)

with open('outputs/analysis/eda_report.json', 'w') as f:
    json.dump(report, f, indent=2, cls=NpEncoder)
print('Saved eda_report.json')

Saved eda_report.json


---
## Part 3: Charts (9 Publication-Quality PNGs)

Each chart is in its own cell below.

### Chart 1: Sector Breakdown

In [7]:
sec_plot = ns24.groupby('sector').agg(tons=('tons_surplus', 'sum'),
    dollars=('us_dollars_surplus', 'sum')).reindex(SECTOR_ORDER)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(sec_plot.index, sec_plot['tons'] / 1e6, color=[COLORS[s] for s in SECTOR_ORDER])
for bar, (_, row) in zip(bars, sec_plot.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'${row["dollars"] / 1e9:.0f}B', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Tons Surplus (Millions)')
ax.set_title('Food Surplus by Sector (2024)', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
sns.despine()
savefig('sector_breakdown.png')

Saved sector_breakdown.png


### Chart 2: Trend Over Time

In [8]:
by_sec = ns.groupby(['year', 'sector'])['tons_waste'].sum().reset_index()
fig, ax = plt.subplots(figsize=(12, 6))
for s in SECTOR_ORDER:
    d = by_sec[by_sec['sector'] == s].sort_values('year')
    ax.plot(d['year'], d['tons_waste'] / 1e6, color=COLORS[s], linewidth=2.5, label=s, marker='o', markersize=4)
ax.axvline(2020, color='red', linestyle='--', alpha=0.6)
ax.annotate('COVID-19', xy=(2020, ax.get_ylim()[1] * 0.95), fontsize=10, color='red', fontweight='bold', ha='center')
ax.set_xlabel('Year')
ax.set_ylabel('Waste (Million Tons)')
ax.set_title('Total Waste by Sector, 2010-2024', fontweight='bold')
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)
sns.despine()
savefig('trend_over_time.png')

Saved trend_over_time.png


### Chart 3: Cause Analysis

In [9]:
cause_sec = nc24.groupby(['cause_name', 'sector'])['tons_surplus_due_to_cause'].sum().reset_index()
top_causes = cause_sec.groupby('cause_name')['tons_surplus_due_to_cause'].sum().nlargest(15).index
cause_top = cause_sec[cause_sec['cause_name'].isin(top_causes)]
dominant = cause_top.loc[cause_top.groupby('cause_name')['tons_surplus_due_to_cause'].idxmax()].set_index('cause_name')['sector']
totals = cause_top.groupby('cause_name')['tons_surplus_due_to_cause'].sum().loc[top_causes].sort_values()
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(totals.index, totals.values / 1e6, color=[COLORS[dominant[c]] for c in totals.index])
ax.set_xlabel('Tons (Millions)')
ax.set_title('Top 15 Causes of Food Surplus (2024)', fontweight='bold')
patches = [mpatches.Patch(color=COLORS[s], label=s) for s in SECTOR_ORDER]
ax.legend(handles=patches, loc='lower right', title='Dominant Sector')
ax.grid(axis='x', alpha=0.3)
sns.despine()
savefig('cause_analysis.png')

Saved cause_analysis.png


### Chart 4: Destination Flow (Stacked Bar)

In [10]:
dest_sec = ns24.groupby('sector')[dest_cols].sum().reindex(SECTOR_ORDER)
dest_labels = [c.replace('tons_', '').replace('_', ' ').title() for c in dest_cols]
fig, ax = plt.subplots(figsize=(14, 7))
bottom = np.zeros(len(SECTOR_ORDER))
cmap = plt.cm.Set3(np.linspace(0, 1, len(dest_cols)))
for i, (col, label) in enumerate(zip(dest_cols, dest_labels)):
    vals = dest_sec[col].values / 1e6
    ax.bar(SECTOR_ORDER, vals, bottom=bottom, label=label, color=cmap[i])
    bottom += vals
ax.annotate('ZERO\nDonations', xy=(SECTOR_ORDER.index('Residential'), 0.1),
            fontsize=9, color='red', fontweight='bold', ha='center')
ax.set_ylabel('Tons (Millions)')
ax.set_title('Waste Destination Breakdown by Sector (2024)', fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.grid(axis='y', alpha=0.3)
sns.despine()
savefig('destination_flow.png')

Saved destination_flow.png


### Chart 5: Food Type Heatmap

In [11]:
wi_df = ns24.groupby(['sector', 'food_type']).agg(w=('tons_waste', 'sum'), s=('tons_supply', 'sum')).reset_index()
wi_df['intensity'] = wi_df['w'] / wi_df['s']
piv = wi_df.pivot(index='food_type', columns='sector', values='intensity').reindex(columns=SECTOR_ORDER).fillna(0)
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(piv, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Waste Intensity'})
ax.set_title('Waste Intensity: Food Type x Sector (2024)', fontweight='bold')
savefig('food_type_heatmap.png')

Saved food_type_heatmap.png


### Chart 6: GHG Impact

In [12]:
ghg = ns24.groupby('sector').agg(co2e=('surplus_total_100_year_mtco2e_footprint', 'sum'),
    tons=('tons_surplus', 'sum')).reindex(SECTOR_ORDER)
ghg['per_ton'] = ghg['co2e'] / ghg['tons']
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.bar(SECTOR_ORDER, ghg['co2e'] / 1e6, color=[COLORS[s] for s in SECTOR_ORDER])
ax1.set_ylabel('CO2e (Million MT)')
ax2 = ax1.twinx()
ax2.plot(SECTOR_ORDER, ghg['per_ton'], 'ko-', linewidth=2, markersize=8)
ax2.set_ylabel('MT CO2e per Ton Surplus')
ax1.annotate(f'Total: {ghg["co2e"].sum() / 1e6:.0f}M MT CO2e\n~{ghg["co2e"].sum() / 4.6 / 1e6:.0f}M cars',
             xy=(0.98, 0.95), xycoords='axes fraction', ha='right', va='top', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax1.set_title('GHG Impact by Sector (2024)', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
sns.despine(right=False)
savefig('ghg_impact.png')

Saved ghg_impact.png


### Chart 7: Economic Value

In [13]:
econ = ns24.groupby(['sector', 'food_type'])['us_dollars_surplus'].sum().reset_index()
econ_piv = econ.pivot(index='sector', columns='food_type', values='us_dollars_surplus').fillna(0).reindex(SECTOR_ORDER)
fig, ax = plt.subplots(figsize=(12, 7))
left = np.zeros(len(SECTOR_ORDER))
cmap2 = plt.cm.tab10(np.linspace(0, 1, len(econ_piv.columns)))
for i, ft_name in enumerate(econ_piv.columns):
    vals = econ_piv[ft_name].values / 1e9
    ax.barh(SECTOR_ORDER, vals, left=left, label=ft_name, color=cmap2[i])
    left += vals
ax.annotate(f'Total: ${ns24["us_dollars_surplus"].sum() / 1e9:.0f}B', xy=(0.98, 0.05),
            xycoords='axes fraction', ha='right', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_xlabel('US Dollars (Billions)')
ax.set_title('Economic Value of Surplus by Sector & Food Type (2024)', fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.grid(axis='x', alpha=0.3)
sns.despine()
savefig('economic_value.png')

Saved economic_value.png


### Chart 8: Donation Gap

In [14]:
dg_plot = ns24.groupby('sector').agg(uneaten=('tons_uneaten', 'sum'),
    donated=('tons_donations', 'sum')).reindex(SECTOR_ORDER)
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(SECTOR_ORDER))
w = 0.35
ax.bar(x - w / 2, dg_plot['uneaten'] / 1e6, w, label='Total Uneaten', color='#E74C3C')
ax.bar(x + w / 2, dg_plot['donated'] / 1e6, w, label='Total Donated', color='#27AE60')
ax.annotate('0 tons donated!\n$141B wasted', xy=(SECTOR_ORDER.index('Residential') + w / 2, 0.05),
            fontsize=10, color='red', fontweight='bold', ha='center')
ax.set_xticks(x)
ax.set_xticklabels(SECTOR_ORDER)
ax.set_ylabel('Tons (Millions)')
ax.set_title('Donation Gap: Uneaten vs Donated (2024)', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
sns.despine()
savefig('donation_gap.png')

Saved donation_gap.png


### Chart 9: COVID Impact

In [15]:
yearly_plot = ns.groupby('year')['tons_surplus'].sum().reset_index().sort_values('year')
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(yearly_plot['year'], yearly_plot['tons_surplus'] / 1e6, 'b-o', linewidth=2.5, markersize=6)
mask = yearly_plot['year'].between(2019, 2021)
ax.fill_between(yearly_plot[mask]['year'], yearly_plot[mask]['tons_surplus'] / 1e6, alpha=0.2, color='red')
ax.plot(yearly_plot[mask]['year'], yearly_plot[mask]['tons_surplus'] / 1e6, 'ro-', linewidth=2.5, markersize=8)
v19 = yearly_plot[yearly_plot['year'] == 2019]['tons_surplus'].values[0]
v20 = yearly_plot[yearly_plot['year'] == 2020]['tons_surplus'].values[0]
v21 = yearly_plot[yearly_plot['year'] == 2021]['tons_surplus'].values[0]
ax.annotate(f'{(v20 - v19) / v19 * 100:.1f}% drop', xy=(2020, v20 / 1e6), xytext=(2020.5, v20 / 1e6 - 1),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='red', fontweight='bold')
ax.annotate(f'+{(v21 - v20) / v20 * 100:.1f}% recovery', xy=(2021, v21 / 1e6), xytext=(2021.5, v21 / 1e6 + 1),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=10, color='green', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Total Surplus (Million Tons)')
ax.set_title('COVID-19 Impact on Food Surplus (2010-2024)', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
sns.despine()
savefig('covid_impact.png')

Saved covid_impact.png


---
## Summary

All outputs generated:
- **eda_report.json** with comprehensive 2024 statistics (overview, sector breakdown, food types, destinations, causes, trends, waste intensity, environmental impact, donation gap, COVID impact)
- **9 charts** saved to `outputs/analysis/charts/`:
  1. sector_breakdown.png
  2. trend_over_time.png
  3. cause_analysis.png
  4. destination_flow.png
  5. food_type_heatmap.png
  6. ghg_impact.png
  7. economic_value.png
  8. donation_gap.png
  9. covid_impact.png